# Module 48 — Marketing Calendar Visualization with Plotly

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Intermediate  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Pandas, Plotly  
> **Prerequisite**: Module 41 (Marketing Mix Modeling)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Marketing Data](#3-synthetic-marketing-data)
4. [Calendar Heatmap](#4-calendar-heatmap)
5. [Gantt Chart](#5-gantt-chart)
6. [Visualization](#6-visualization)
7. [Key Business Takeaway](#7-key-business-takeaway)

---
## §1 · Business Context

### Why visualize marketing calendar?

Marketing calendar visualization **improves planning and coordination**:
- **Visibility**: See all campaigns at a glance.
- **Coordination**: Avoid overlapping campaigns.
- **Optimization**: Identify gaps and opportunities.

**Applications**:
1. **Campaign planning**: Schedule campaigns across channels.
2. **Budget allocation**: Visualize spend over time.
3. **Performance tracking**: Overlay revenue with campaigns.

In this notebook we will:
1. Generate synthetic marketing data over 52 weeks.
2. Create calendar heatmap of spend vs revenue.
3. Build Gantt chart of campaign timelines.
4. Overlay holidays and seasonality markers.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Visualization ────────────────────────────────────────────────
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")

print("✓ All imports loaded")

---
## §3 · Synthetic Marketing Data

In [ ]:
# Generate synthetic marketing data over 52 weeks
N_WEEKS = 52

data = {
    'week': pd.date_range('2024-01-01', periods=N_WEEKS, freq='W'),
    'tv_spend': np.random.uniform(50000, 200000, N_WEEKS),
    'social_spend': np.random.uniform(20000, 80000, N_WEEKS),
    'search_spend': np.random.uniform(30000, 100000, N_WEEKS),
    'email_spend': np.random.uniform(5000, 20000, N_WEEKS),
}

df = pd.DataFrame(data)
df['total_spend'] = df[['tv_spend', 'social_spend', 'search_spend', 'email_spend']].sum(axis=1)

# Generate revenue (correlated with spend + seasonality)
seasonality = 50000 * np.sin(np.linspace(0, 4 * np.pi, N_WEEKS))
df['revenue'] = 500000 + 0.3 * df['total_spend'] + seasonality + np.random.normal(0, 30000, N_WEEKS)

# Add campaigns
campaigns = [
    {'name': 'Summer Sale', 'start_week': 20, 'end_week': 25, 'channel': 'TV'},
    {'name': 'Back to School', 'start_week': 30, 'end_week': 35, 'channel': 'Social'},
    {'name': 'Holiday Campaign', 'start_week': 45, 'end_week': 52, 'channel': 'All'},
    {'name': 'Spring Promotion', 'start_week': 10, 'end_week': 15, 'channel': 'Email'},
]

print(f"✓ Generated {N_WEEKS} weeks of marketing data")
print(f"  Total spend: ${df['total_spend'].sum():,.0f}")
print(f"  Total revenue: ${df['revenue'].sum():,.0f}")
print(f"  Campaigns: {len(campaigns)}")

---
## §4 · Calendar Heatmap

In [ ]:
# Create calendar heatmap
df['month'] = df['week'].dt.month
df['week_of_month'] = df['week'].dt.isocalendar().week % 4

# Pivot for heatmap
heatmap_data = df.pivot_table(
    values='revenue',
    index='month',
    columns='week_of_month',
    aggfunc='mean'
)

fig = px.imshow(
    heatmap_data,
    labels=dict(x="Week of Month", y="Month", color="Revenue ($)") ,
    title='Revenue Heatmap by Month and Week',
    color_continuous_scale='Viridis',
    aspect='auto'
)

fig.update_layout(height=400)
fig.show()

print("💡 Heatmap shows revenue patterns by month and week")

---
## §5 · Gantt Chart

In [ ]:
# Create Gantt chart for campaigns
gantt_data = []
for campaign in campaigns:
    gantt_data.append({
        'Task': campaign['name'],
        'Start': df['week'].iloc[campaign['start_week']],
        'Finish': df['week'].iloc[campaign['end_week']],
        'Resource': campaign['channel']
    })

df_gantt = pd.DataFrame(gantt_data)

fig = px.timeline(
    df_gantt,
    x_start='Start',
    x_end='Finish',
    y='Task',
    color='Resource',
    title='Marketing Campaign Timeline'
)

fig.update_layout(height=300)
fig.show()

print("💡 Gantt chart shows campaign timing and channel allocation")

---
## §6 · Visualization

In [ ]:
# Combined visualization
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Weekly Spend vs Revenue', 'Channel Spend Breakdown'],
    shared_xaxes=True,
    vertical_spacing=0.1
)

# Spend vs Revenue
fig.add_trace(go.Bar(
    x=df['week'],
    y=df['total_spend'],
    name='Total Spend',
    marker_color='#EF553B'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df['week'],
    y=df['revenue'],
    name='Revenue',
    mode='lines',
    line=dict(color='#636EFA', width=3)
), row=1, col=1)

# Channel breakdown
for channel in ['tv_spend', 'social_spend', 'search_spend', 'email_spend']:
    fig.add_trace(go.Scatter(
        x=df['week'],
        y=df[channel],
        name=channel.replace('_spend', '').title(),
        stackgroup='one'
    ), row=2, col=1)

fig.update_layout(height=600, showlegend=True)
fig.show()

print("💡 Combined view shows spend, revenue, and channel mix over time")

---
## §7 · Key Business Takeaway

> ### 🎯 Action Items for Marketing Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Campaign planning, performance reporting, coordination |
> | **Visualization** | Heatmaps for overview, Gantt for timelines |
> | **Metrics** | Spend, revenue, ROI by week/month |
> | **Overlay** | Holidays, campaigns, seasonality |
> | **Frequency** | Weekly review, monthly planning |
>
> ### 🔑 Key Takeaways
>
> 1. **Calendar visualization improves marketing coordination**.
> 2. **Heatmaps reveal seasonal patterns** in performance.
> 3. **Gantt charts show campaign timing** and overlaps.
> 4. **Overlaying spend and revenue** shows ROI patterns.
> 5. **Regular visualization** enables better planning decisions.